## Packages Import


In [17]:
import os
import re
import yaml
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

## Apollo Scraper

In [18]:
with open("config.yaml", "r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config['credentials']['user']
password = config['credentials']['password']
credentials = HTTPBasicAuth(username, password)

In [19]:
group_id = input("Enter group ID: ")
url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={group_id}&okres=1"
response = requests.get(url, auth=credentials)
response.encoding = "UTF-8"
print(response.status_code)


200


In [20]:
print(response.status_code)
print(response.text[:500])

200
<!DOCTYPE html PUBLIC "-//W3C//DTD HTML 4.0 Transitional//EN" "http://www.w3.org/TR/REC-html40/loose.dtd">
<html>
<head>
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
<title>Plan zajęć UEK ZICSS1-1211</title>
<link rel="stylesheet" type="text/css" href="planzajec.css">
</head>
<body>
<script src="js/accessibility_uek.js"></script><script src="js/apollo2calendar.js"></script><div class="naglowek">
<div class="logo"><img src="UEK-logo.gif" alt=""></div>
<div class="planzajec"


In [21]:
page_dom = BeautifulSoup(response.text, 'html.parser')


In [22]:
group = page_dom.select_one("div.grupa")
if group:
    group = group.get_text(strip=True)
else:
    group = "unknown"
print(group)

ZICSS1-1211


### TABLE DATA FRAME

In [23]:
classes_tag = page_dom.select_one("table")
with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(classes_tag.prettify())
classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

### FILTER

In [24]:
classes = classes.loc[classes['Typ'].isin(["ćwiczenia", "wykład", "egzamin"])]

### SPLIT DAY TIME

In [25]:
split_cols = classes['Dzień, godzina'].str.split(' ', expand=True)
print(split_cols.shape)
print(split_cols.head())

(17, 5)
     0      1  2      3      4
1   Pn  13:15  -  15:45  (3g.)
4   Wt  09:45  -  11:15  (2g.)
8   Wt  15:00  -  16:30  (2g.)
9   Wt  18:30  -  20:00  (2g.)
11  Śr  13:15  -  14:45  (2g.)


### CLEAN SALA

In [26]:
classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.).+",
    r"\1",
    regex=True
)

###  EXPORT

In [27]:
if not os.path.exists("schedules"):
    os.makedirs("schedules")
classes.to_csv(f"schedules/{group}.csv")

classes

,Termin,"Dzień, godzina",Przedmiot,Typ,Nauczyciel,Sala
1,2026-05-18,Pn 13:15 - 15:45 (3g.),Computer Programming 2,ćwiczenia,dr Katarzyna Wójcik,Paw.A 013 lab.
4,2026-05-19,Wt 09:45 - 11:15 (2g.),Probability and Statistics,wykład,prof. dr hab. Andrzej Sokołowski,Paw.F 008
8,2026-05-19,Wt 15:00 - 16:30 (2g.),Discrete mathematics,wykład,dr Grzegorz Kosiorowski,Rakowicka 16 sala 11
9,2026-05-19,Wt 18:30 - 20:00 (2g.),Business Law,wykład,dr Jacek Lachner,Rakowicka 16 sala 11
11,2026-05-20,Śr 13:15 - 14:45 (2g.),Operating Systems and Computer Networks,ćwiczenia,prof. UEK dr hab. Joanna Wyrobek,Paw.A 014 lab.
13,2026-05-20,Śr 15:00 - 16:30 (2g.),Discrete mathematics,ćwiczenia,dr Igor Sikora,Paw.U sala N
15,2026-05-21,Cz 08:00 - 09:30 (2g.),Operating Systems and Computer Networks,wykład,prof. UEK dr hab. Joanna Wyrobek,Paw.F 008
19,2026-05-21,Cz 13:15 - 14:45 (2g.),Discrete mathematics,ćwiczenia,dr Igor Sikora,Paw.B 052 (KSB)
21,2026-05-21,Cz 15:00 - 16:30 (2g.),Operating Systems and Computer Networks,ćwiczenia,prof. UEK dr hab. Joanna Wyrobek,Paw.A 07 lab.
23,2026-05-21,Cz 16:45 - 18:15 (2g.),Information Systems,ćwiczenia,mgr inż. Justyna Olczak,Bibl.425
